In [ ]:
# Load in required libraries for analysis
library(tidyr)
library(ggplot2)
library(ggh4x)
library(dplyr)
library(RColorBrewer)
library(tidyselect)
library(tidyr)
library(stringr)
library(ggpubr)
library(ggdist)
library(ggforce)
library(labdsv)
library(vegan)
library(readr)
library(ape)
library(lme4)
library(lsmeans)
library(scales)
library(purrr)
library(broom)
library(Polychrome)
library(cowplot)
library(reshape2)
library(doParallel)

In [ ]:
metadata <- read.csv("DOME-16S-Nasal-Metadata.csv")
metadata <- metadata %>%
  dplyr::mutate(Sample.ID = stringr::str_remove(Sample.ID, "^19-"))

GENUS <- read.csv("DOME-16S-Taxa-Counts-Genus.csv")

asv_counts <- GENUS %>% dplyr::rename(Sample.ID = X)

genus_colors <- c("Acinetobacter" = "#C6946C",
                  "Aerococcus" = "#C09B00",
                  "Arthrobacter_B" = "#BA9F00",
                  "Bacillus" = "#32D4C7",
                  "Bacillus_A" = "#B1A300",
                  "Brachybacterium" = "#C9CB81",
                  "Brevibacillus" = "#BDCD84",
                  "Brevibacterium" = "#C3CC7E",
                  "Cellulosimicrobium" = "#9F00BB",
                  "Citricoccus" = "#7EAD00",
                  "Corynebacterium" = "#93D19C",
                  "Cytobacillus" = "#00B27D",
                  "Dolosigranulum" = "#00B393",
                  "Kocuria" = "#DCB9E6",
                  "Lysinibacillus" = "#B4006C",
                  "Mammaliicoccus" = "#D33639",
                  "Microbacterium" = "#00B2BA",
                  "Micrococcus" = "#89D2A1",
                  # "Psychrobacter" = "#FDE4CF",
                  "Nocardiopsis" = "#E8B2D5",
                  "Paenibacillus" = "#00AFC2",
                  "Priestia" = "#00ACC9",
                  "Shouchella" = "#B3C6E9",
                  "Staphylococcus" = "#E8B8B5",
                  "Streptomyces" = "#00A7CD",
                  "Virgibacillus" = "#91A0D0",
                  "Other" = "#B5B5B2",
                    "Cow" = "#990006",
                    "Non-Farmer" = "#B5B5B2", 
                    "Farmer" = "#386EC2")

In [ ]:
asv_counts <- asv_counts %>%
tibble::column_to_rownames("Sample.ID") %>%
 select(where(is.numeric))

In [ ]:
richness <- specnumber(asv_counts)

In [ ]:
write.csv(asv_counts, "16S-Nasal-ASV-Counts.csv", row.names = TRUE)
write.csv(richness, "16S-Nasal-Richness.csv", row.names = TRUE)

In [ ]:
metadata <- metadata %>% mutate(Sample.ID = as.character(Sample.ID))
richness_meta <- metadata %>% mutate(Richness = richness[Sample.ID])

In [ ]:
richness_meta <- richness_meta %>%
mutate(
    Season = factor(
      Season,
      levels = c("Spring", "Summer", "Autumn", "Winter")
    )
)

In [ ]:
write.csv(richness_meta, "16S-Nasal-Richness-With-Metadata.csv", row.names = FALSE)

In [ ]:
richness_plot <- ggplot(richness_meta, aes(x = Cohort, y = Richness, fill = Cohort)) +
  # Boxplot
  geom_boxplot(
    outlier.shape = NA,
    alpha = 0.8,
    color = "black",
    width = 0.5,
    position = position_dodge(width = 0.8)
  ) +
guides(fill = guide_legend(override.aes = list(color = NA, alpha = 1))) +
  # Half-eye (raincloud)
  stat_halfeye(
    aes(fill = Cohort),
    alpha = 0.4,
    width = 0.6,
    justification = -0.25,
    point_colour = NA,
    position = position_dodge(width = 0.8)
  ) +
  # Jittered points
  geom_jitter(
    aes(color = Cohort),   
    size = .25,
    alpha = 0.75,
    position = position_jitterdodge(dodge.width = 0.8, jitter.width = 0.15)
  ) +
  scale_fill_manual(values = genus_colors) +
  scale_color_manual(values = genus_colors) +
  facet_grid2(Season ~ Year, scales = "free_y", strip = ggh4x::strip_vanilla()) +
  theme_bw() +
  labs(y = "Richness", x = "Cohort") +
  theme(
    panel.grid = element_blank(),               
    strip.background = element_rect(fill = "black"),
    strip.text = element_text(color = "white", face = "bold", size = 12),
    axis.text.x = element_text(size = 8),
    axis.title.x = element_blank(),
    text = element_text(family = "Helvetica", size = 12)
  ) 

In [ ]:
ggplot2::ggsave(filename = "DOME-16S-Richness.pdf", richness_plot, width = 10, height =  8, dpi = 320, unit = "in")

In [ ]:
# Modifying metadata for Wilcoxon testing
richness_meta <- richness_meta %>%
    mutate(Group = paste(Season, Year, Cohort, sep = "_"))

In [ ]:
unique_groups <- unique(richness_meta$Group)
pairwise_comparisons <- t(combn(unique_groups, 2))
if (is.null(dim(pairwise_comparisons))) {
  pairwise_comparisons <- matrix(pairwise_comparisons, nrow = 1)
}

In [ ]:
results <- data.frame(Group1=character(), Group2=character(),
                      Statistic=numeric(), P_value=numeric(), stringsAsFactors=FALSE)

for (i in seq_len(nrow(pairwise_comparisons))) {
  g1 <- pairwise_comparisons[i, 1]
  g2 <- pairwise_comparisons[i, 2]
  
  subset_data <- dplyr::filter(richness_meta, Group %in% c(g1, g2))
  test <- wilcox.test(Richness ~ Group, data = subset_data, exact = FALSE)
  
  results <- rbind(results, data.frame(
    Group1 = g1,
    Group2 = g2,
    P_value = test$p.value,
    stringsAsFactors = FALSE
  ))
}

results$Adjusted_P <- p.adjust(results$P_value, method = "BH")
results <- results %>%
  mutate(Significance = case_when(
    Adjusted_P < 0.0001 ~ "****",
    Adjusted_P < 0.001  ~ "***",
    Adjusted_P < 0.01   ~ "**",
    Adjusted_P < 0.05   ~ "*",
    TRUE                ~ "ns"
  ))

In [ ]:
write.csv(results, "16S-Richness-Wilcoxon-Testing.csv", row.names = FALSE)

In [ ]:
significant_results <- results %>%
filter(Adjusted_P < .05)
write.csv(significant_results, "16S-Richness-Wilcoxon-Testing-Significant-Only.csv", row.names = FALSE)